# **Initialization**

In [1]:
print('Start')

Start


In [2]:
#%load_ext autoreload
%reload_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import math
import random
import sys
import re
import sys
import os
import gc
import contextlib
import modified_didppy as m_dp
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.optimize import linear_sum_assignment
from numpy.linalg import eigh
import time as pytime
import pulp
from ortools.linear_solver import pywraplp
import requests

# --- 1. DEFINE PATH TO LIBRARY PARENT FOLDER ---
# Replace this with the ACTUAL path to the folder containing 'didp_ea_lib'
# IMPORTANT: Use r"..." string to handle Windows backslashes correctly
LIBRARY_PARENT_PATH = r"C:\Users\ACER\Desktop\Code\0.Thesis implementation\2_DIDP_custom_search_guidance_local\THESIS_MODIFIED_DIDP\Evolutionary_algorithm"
# --- 2. ADD TO SYSTEM PATH ---
if LIBRARY_PARENT_PATH not in sys.path:
    sys.path.append(LIBRARY_PARENT_PATH)
print(f"Library path added: {LIBRARY_PARENT_PATH}")
# --- 3. TEST IMPORT ---
try:
    import evolutionary_algorithm_lib
    from evolutionary_algorithm_lib import *
    from evolutionary_algorithm_lib import (compile_chromosome_to_useable_function, 
                                            combining_modified_didppy_solver_with_chromosome)
    from evolutionary_algorithm_lib.utils import automatic_creation_of_dual_bounds_registry
    
    print("✅ Success! 'evolutionary_algorithm_lib' is imported and ready.")
except ImportError as e:
    print(f"❌ Error: Could not import library. Check the path above.\nDetails: {e}")

Library path added: C:\Users\ACER\Desktop\Code\0.Thesis implementation\2_DIDP_custom_search_guidance_local\THESIS_MODIFIED_DIDP\Evolutionary_algorithm
✅ Success! 'evolutionary_algorithm_lib' is imported and ready.


# **Data**

In [3]:
# =========================================================
# 1. LOADER FUNCTION (IO Operations)
# =========================================================
def load_jsp_file(file_path):
    """
    Reads the content of a JSP instance file from disk.
    
    Args:
        file_path (str): The absolute or relative path to the .txt file.
        
    Returns:
        str: The raw string content of the file.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"The file was not found at: {file_path}")
    
    with open(file_path, 'r') as f:
        content = f.read()
    
    return content

In [5]:
def parse_jsp_instance(file_content):
    """
    Parses a JSP instance string (Taillard/Beasley format) and generates 
    ALL sets/parameters required for both LP Relaxation and DIDP models.
    
    Args:
        file_content (str): The raw text content of the instance file.
        
    Returns:
        dict: A dictionary containing all sets (J, M, N, A, B, etc.) and parameters.
    """
    # ---------------------------------------------------------
    # 1. Basic Parsing (Clean comments and header)
    # ---------------------------------------------------------
    lines = file_content.strip().split('\n')
    data_tokens = []
    
    for line in lines:
        # Remove comments (lines starting with # or [)
        if not line.strip() or line.strip().startswith('#') or line.strip().startswith('['):
            continue
        # Extract numbers
        tokens = re.findall(r'\d+', line)
        data_tokens.extend([int(t) for t in tokens])
        
    iterator = iter(data_tokens)
    
    try:
        num_jobs = next(iterator)
        num_machines = next(iterator)
    except StopIteration:
        raise ValueError("File is empty or invalid format")

    # ---------------------------------------------------------
    # 2. Raw Job Data Extraction
    # ---------------------------------------------------------
    # jobs_data format: [ [(machine, time), (machine, time)], ... ]
    jobs_data = []
    for _ in range(num_jobs):
        job_seq = []
        for _ in range(num_machines):
            m = next(iterator)
            p = next(iterator)
            job_seq.append((m, p))
        jobs_data.append(job_seq)

    num_operations = num_jobs * num_machines

    # ---------------------------------------------------------
    # 3. Build Structures for LP Relaxation (Lecture Note Style)
    # ---------------------------------------------------------
    # Sets based on (machine, job) tuples
    N = []              # Nodes: List of (m, j)
    A = []              # Solid Arcs: List of ((m_prev, j), (m_curr, j))
    B = []              # Broken Arcs (Graph): List of ((m, j), (m, k))
    B_flat = []         # Broken Arcs (LP Triplet): List of (m, j, k)
    p_mj = {}           # Processing time map: {(m, j): time}
    
    # Helpers for Broken Arcs
    machine_allocations = {m: [] for m in range(num_machines)}

    for j_idx, job_seq in enumerate(jobs_data):
        prev_node = None
        for (m, p) in job_seq:
            curr_node = (m, j_idx)
            
            # Populate N and p_mj
            N.append(curr_node)
            p_mj[curr_node] = float(p)
            
            # Populate Solid Arcs (A)
            if prev_node is not None:
                A.append((prev_node, curr_node))
            
            # Track for Broken Arcs
            machine_allocations[m].append(j_idx)
            prev_node = curr_node

    # Populate Broken Arcs - pairs on same machine
    for m in range(num_machines):
        assigned_jobs = machine_allocations[m]
        for i in range(len(assigned_jobs)):
            for k in range(i + 1, len(assigned_jobs)):
                job_1 = assigned_jobs[i]
                job_2 = assigned_jobs[k]
                
                # 1. Standard Disjunctive Graph Arc: ((m, j), (m, k))
                # Useful for graph plotting and disjunctive graph algos
                B.append(((m, job_1), (m, job_2)))
                
                # 2. Flat Triplet for LP variables: (m, j, k)
                # Useful for PuLP variables x_mjk
                B_flat.append((m, job_1, job_2))

    # ---------------------------------------------------------
    # 4. Build Structures for DIDP (Operation Index Style)
    # ---------------------------------------------------------
    # Flattened Operation ID: 0 to num_operations-1
    op_job_type = []
    op_required_machine_type = []
    op_processing_time = []
    op_predecessors = []
    op_deadline = []
    op_same_job = [[] for _ in range(num_operations)] # SJ_jo
    ops_on_machine = [[] for _ in range(num_machines)] # AO_k
    
    # Mapping to convert between (m,j) and op_id
    map_mj_to_op = {}
    
    # Define a default deadline (e.g., a large number covering the horizon)
    DEFAULT_DEADLINE = float('inf')
    
    op_counter = 0
    for j_idx, job_seq in enumerate(jobs_data):
        job_ops_indices = []
        
        for i, (m, p) in enumerate(job_seq):
            # Basic Attributes
            op_job_type.append(j_idx)
            op_required_machine_type.append(m)
            op_processing_time.append(float(p))
            op_deadline.append(DEFAULT_DEADLINE)
            
            # Map tracking
            map_mj_to_op[(m, j_idx)] = op_counter
            ops_on_machine[m].append(op_counter)
            job_ops_indices.append(op_counter)
            
            # Predecessors (P_jo)
            if i > 0:
                op_predecessors.append([op_counter - 1])
            else:
                op_predecessors.append([]) # First op has no pred
            
            op_counter += 1
            
            
        # Same Job Set (SJ_jo)
        for o_id in job_ops_indices:
            # All ops in this job EXCEPT self
            op_same_job[o_id] = [x for x in job_ops_indices if x != o_id]

    # Valid Machines (VM_jo) - Simple for static JSP
    valid_machines = [[op_required_machine_type[o]] for o in range(num_operations)]

    # ---------------------------------------------------------
    # 5. Return Master Dictionary
    # ---------------------------------------------------------
    return {
        "metadata": {
            "num_jobs": num_jobs,
            "num_machines": num_machines,
            "num_operations": num_operations
        },
        
        # --- Mathematical Programming Sets (LP Relax) ---
        "LP_sets": {
            "N": N,         # Nodes (m, j)
            "A": A,         # Solid Arcs
            "B": B,         # Broken Arcs (Nested Tuples)
            "B_flat": B_flat, # Broken Arcs (Flat Triplets: m, j, k)
            "p_mj": p_mj,   # Parameter p
            "J": list(range(num_jobs)),
            "M": list(range(num_machines))
        },
        
        # --- DIDP State Sets ---
        "DIDP_sets": {
            "op_job_type": op_job_type,                 # Job J per op
            "op_required_machine_type": op_required_machine_type, # Machine M per op
            "op_processing_time": op_processing_time,   # Time p per op
            "op_predecessors": op_predecessors,        # P_jo
            "op_deadline": op_deadline,               # Deadline D_jo
            "op_same_job": op_same_job,                 # SJ_jo
            "ops_on_machine": ops_on_machine,           # AO_k
            "valid_machines": valid_machines,           # VM_jo
            "map_mj_to_op": map_mj_to_op                # Helper: (m,j) -> op_id
        }
    }

In [6]:
# =========================================================
# 3. EXECUTION & VARIABLE EXTRACTION
# =========================================================
data_file_path = r"C:\Users\ACER\Desktop\Code\0.Thesis implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\5_JSP_dual_bounds_and_models\ft20.txt"
try:
    # 1. Load and Parse
    content = load_jsp_file(data_file_path)
    current_data = parse_jsp_instance(content)
    print("Successfully parsed instance!")
    print(f"Jobs: {current_data['metadata']['num_jobs']}, Machines: {current_data['metadata']['num_machines']}")

    
    # 2. Extract basic data
    current_number_of_operations = current_data['metadata']['num_operations']
    current_number_of_machines = current_data['metadata']['num_machines']
    current_number_of_jobs = current_data['metadata']['num_jobs']
    
    # 3. Extract LP Relaxation Variables
    lp_data = current_data['LP_sets']
    current_N = lp_data['N']          # Nodes
    current_A = lp_data['A']          # Solid Arcs
    current_B = lp_data['B']          # Broken Arcs (Nested Tuples)
    current_B_flat = lp_data['B_flat']# Broken Arcs (Flat Tuples)
    current_p_mj = lp_data['p_mj']    # Processing Times
    current_J = lp_data['J']          # Set of Jobs
    current_M = lp_data['M']          # Set of Machines

    # 3. Extract DIDP Variables
    didp_data = current_data['DIDP_sets']
    current_op_job_type = didp_data['op_job_type']
    current_op_required_machine_type = didp_data['op_required_machine_type']
    current_op_processing_time = didp_data['op_processing_time']
    current_op_deadline = didp_data['op_deadline']
    current_op_predecessors = didp_data['op_predecessors']
    current_op_same_job = didp_data['op_same_job']
    current_ops_on_machine = didp_data['ops_on_machine']
    current_valid_machines = didp_data['valid_machines']
    current_map_mj_to_op = didp_data['map_mj_to_op']

    # 4. Verify Extraction
    print("\n--- LP Variables Extracted ---")
    print(f"N: {len(current_N)} nodes")
    print(f"B_flat: {len(current_B_flat)} disjunctive triplets")
    
    print("\n--- DIDP Variables Extracted ---")
    print(f"op_processing_time: {len(current_op_processing_time)} entries")
    print(f"ops_on_machine[0]: {len(current_ops_on_machine[0])} operations on Machine 0")
    print(f"Processingtime of operations:{current_op_processing_time}")

except FileNotFoundError:
    print(f"Error: The file was not found at {data_file_path}")
except Exception as e:
    print(f"An error occurred: {e}")

Successfully parsed instance!
Jobs: 20, Machines: 5

--- LP Variables Extracted ---
N: 100 nodes
B_flat: 950 disjunctive triplets

--- DIDP Variables Extracted ---
op_processing_time: 100 entries
ops_on_machine[0]: 20 operations on Machine 0
Processingtime of operations:[29.0, 9.0, 49.0, 62.0, 44.0, 43.0, 75.0, 69.0, 46.0, 72.0, 91.0, 39.0, 90.0, 12.0, 45.0, 81.0, 71.0, 9.0, 85.0, 22.0, 14.0, 22.0, 26.0, 21.0, 72.0, 84.0, 52.0, 48.0, 47.0, 6.0, 46.0, 61.0, 32.0, 32.0, 30.0, 31.0, 46.0, 32.0, 19.0, 36.0, 76.0, 76.0, 85.0, 40.0, 26.0, 85.0, 61.0, 64.0, 47.0, 90.0, 78.0, 36.0, 11.0, 56.0, 21.0, 90.0, 11.0, 28.0, 46.0, 30.0, 85.0, 74.0, 10.0, 89.0, 33.0, 95.0, 99.0, 52.0, 98.0, 43.0, 6.0, 61.0, 69.0, 49.0, 53.0, 2.0, 95.0, 72.0, 65.0, 25.0, 37.0, 13.0, 21.0, 89.0, 55.0, 86.0, 74.0, 88.0, 48.0, 79.0, 69.0, 51.0, 11.0, 89.0, 74.0, 13.0, 7.0, 76.0, 52.0, 45.0]


# **1. Model and dual bound declaration**

In [7]:
def creation_of_didp_model_function():
    num_operations = current_number_of_operations
    num_machines = current_number_of_machines
    op_processing_time = current_op_processing_time
    op_deadline = current_op_deadline
    op_predecessors = current_op_predecessors
    op_same_job = current_op_same_job
    valid_machines = current_valid_machines
    ops_on_machine = current_ops_on_machine
    # -----------------------------
    # DIDP Model and Constants
    # -----------------------------
    model = m_dp.Model(maximize=False, float_cost=True)

    operation_obj = model.add_object_type(number=num_operations)
    machine_obj = model.add_object_type(number=num_machines)

    # Constant tables
    processing_time_table = model.add_float_table(op_processing_time)
    deadline_table = model.add_float_table(op_deadline)
    predecessor_table = model.add_set_table(op_predecessors, object_type=operation_obj)
    same_job_table = model.add_set_table(op_same_job, object_type=operation_obj)
    valid_machine_table = model.add_set_table(valid_machines, object_type=operation_obj)

    # -----------------------------
    # State Variables
    # -----------------------------
    unscheduled_operations = model.add_set_var(object_type=operation_obj,
                                    target=list(range(num_operations)))
    finished_operations = model.add_set_var(object_type=operation_obj, target=[])
    machine_available_time = [
        model.add_float_resource_var(target=0, less_is_better=True, name=f"at_{mi}")
        for mi in range(num_machines)
    ]
    op_completion_time = [
        model.add_float_resource_var(target=0, less_is_better=True, name=f"c_{jo}")
        for jo in range(num_operations)
    ]

    # ------------------------------------------------------------
    # Transition
    # ------------------------------------------------------------
    # Schedule operation jo on the specified machine
    # ------------------------------------------------------------
    for o in range(num_operations):
        valid_machines_for_op = valid_machines[o]
        expr = m_dp.FloatExpr(0)
        #Finding the completion time of the predecessors to make sure 1 job cannot be operated on 2 machines at the same time
        if op_same_job[o]:
            job_exprs = [op_completion_time[k] for k in op_same_job[o]]
            expr = job_exprs[0]
            for e in job_exprs[1:]:
                expr = m_dp.max(expr, e)
        max_same_job_completion_time = expr
        for m in valid_machines_for_op:
            # Compute start & completion
            start_time = m_dp.max(machine_available_time[m], max_same_job_completion_time)
            completion_time = start_time + processing_time_table[o]
            cost_expr = m_dp.FloatExpr.state_cost()
            transition = m_dp.Transition(
                name=f"schedule_op{o}_on_m{m}",
                cost=cost_expr,
                preconditions=[
                    unscheduled_operations.contains(o),
                    predecessor_table[o].issubset(finished_operations),
                    completion_time <= deadline_table[o],
                ],
                effects=[
                    (unscheduled_operations, unscheduled_operations.remove(o)),
                    (finished_operations, finished_operations.add(o)),
                    (op_completion_time[o], completion_time),
                    (machine_available_time[m], completion_time),
                ],
            )
            model.add_transition(transition)

    # ------------------------------------------------------------
    # Base case
    # ------------------------------------------------------------
    makespan = m_dp.FloatExpr(0)
    for o in range(num_operations):
        makespan = m_dp.max(makespan, op_completion_time[o])
    model.add_base_case([unscheduled_operations.is_empty()], cost=makespan)

    # State Constraints
    # ============================================================
    # Enforce 0 <= completion time <= deadline for all operations
    # ============================================================
    for o in range(num_operations):
        # Upper bound: c(o) <= d(o)
        model.add_state_constr(op_completion_time[o] <= deadline_table[o])

    # ============================================================
    # Dual Bound (optional)
    # ============================================================
    # Machine-based bound: for each machine m, available_time[m] + sum(remaining ptime on that machine)
    ops_on_machine_consts = [
        model.create_set_const(object_type=operation_obj, value=ops) for ops in ops_on_machine
    ]
    # remaining processing time on machine m = sum of ptime for operations on m that are still unscheduled
    remaining_time_on_machine = [
        processing_time_table[unscheduled_operations.intersection(ops_on_machine_consts[m])]
        for m in range(num_machines)
    ]
    machine_bound_exprs = [
        machine_available_time[m] + remaining_time_on_machine[m]
        for m in range(num_machines)
    ]

    # fold to a single dp.max expression
    dual_bound_expr = machine_bound_exprs[0]
    for b in machine_bound_exprs[1:]:
        dual_bound_expr = m_dp.max(dual_bound_expr, b)

    # Add dual bound to model
    model.add_dual_bound(dual_bound_expr)

    # =========================================================
    # 3. Bundle Metadata
    # =========================================================
    metadata = {
        "unscheduled_operations": unscheduled_operations,
        "finished_operations": finished_operations,
        "machine_available_time": machine_available_time,
        "op_completion_time": op_completion_time,
    }
    
    didp_bundle = (model, metadata)
    return didp_bundle

In [ ]:
def create_persistent_lp_relaxation_standard_jsp(didp_bundle):
    """
    Creates a persistent LP model using the standard (Machine, Job) notation.
    - Matches the PDF formulation: Nodes (m,j), Solid Arcs A, Broken Arcs B.
    - Bridges DIDP state (op_id) to LP variables (m,j) dynamically.
    """
    model_ref, metadata = didp_bundle
    
    # 1. Extract Standard LP Sets (Matches PDF Step 2)
    N = current_N          # Nodes
    A = current_A          # Solid Arcs
    B = current_B          # Broken Arcs (Nested Tuples)
    p_mj = current_p_mj    # Processing Times
    M = current_M          # Set of Machines
    J = current_J          # Set of Jobs
    
    # Helpers for Transformation (using Globals)
    op_job = current_op_job_type               
    op_machine = current_op_required_machine_type 
    num_ops = current_number_of_operations
    
    # State keys
    unscheduled_var = metadata['unscheduled_operations']
    machine_at_vars = metadata['machine_available_time']
    op_c_vars = metadata['op_completion_time']

    # Big-M: Sufficiently large number
    # Sum of all processing times is a safe upper bound
    BIG_M = sum(p_mj.values()) * 1.5

    # ==========================================
    # 3. INITIALIZATION (Runs Once)
    # ==========================================
    solver = pywraplp.Solver.CreateSolver('GLOP')
    if not solver:
        return lambda state: 0.0
    
    infinity = solver.infinity()

    # --- Create Variables ---
    
    # y[(m,j)]: Start time of job j on machine m (Continuous, >= 0)
    y = {}
    for node in N:
        m, j = node
        y[node] = solver.NumVar(0, infinity, f'y_{m}_{j}')

    # Cmax: Makespan variable
    c_max = solver.NumVar(0, infinity, 'Cmax')

    # x[(m,j,k)]: Binary variables for disjunctive pairs
    # x_mjk = 1 if job j precedes job k on machine m
    x = {} 
    
    # --- Create Constraints ---

    # 1. Job Precedence Constraints (Solid Arcs A)
    # y_hj >= y_mj + p_mj  => y_hj - y_mj >= p_mj
    for arc in A:
        node_mj, node_hj = arc
        
        # y_hj - y_mj >= p_mj
        c_prec = solver.Constraint(p_mj[node_mj], infinity, f'prec_{node_mj}_{node_hj}')
        c_prec.SetCoefficient(y[node_hj], 1)
        c_prec.SetCoefficient(y[node_mj], -1)

    # 2. Machine Capacity Constraints (Broken Arcs B)
    # For every pair of operations on the same machine, ensure disjointness.
    # Logic: B contains ((m,j), (m,k)). We need to define x_mjk and x_mkj.
    
    processed_pairs = set()
    
    for pair in B:
        # pair is ((m,j), (m,k))
        node_j, node_k = pair 
        m = node_j[0] # Extract machine index m
        j = node_j[1] # Job j
        k = node_k[1] # Job k
        
        # Ensure we only process the set {j, k} once per machine
        sorted_pair = tuple(sorted((j, k)))
        unique_key = (m, sorted_pair)
        
        if unique_key not in processed_pairs:
            processed_pairs.add(unique_key)
            
            # --- Define Variables x_mjk and x_mkj ---
            # Relaxed binary variables: 0 <= x <= 1
            x_mjk_key = (m, j, k)
            x_mkj_key = (m, k, j)
            
            x[x_mjk_key] = solver.NumVar(0, 1, f'x_{m}_{j}_{k}')
            x[x_mkj_key] = solver.NumVar(0, 1, f'x_{m}_{k}_{j}')
            
            # --- Symmetry Constraint ---
            # x_mjk + x_mkj = 1
            c_sym = solver.Constraint(1, 1, f'sym_{m}_{j}_{k}')
            c_sym.SetCoefficient(x[x_mjk_key], 1)
            c_sym.SetCoefficient(x[x_mkj_key], 1)

            # --- Disjunctive Constraints (Big-M) ---
            # Constraint 1: If j -> k (x_mjk=1), then y_mk >= y_mj + p_mj
            # Formula: y_mk >= y_mj + p_mj - M * (1 - x_mjk)
            # Rearranged: y_mk - y_mj - M * x_mjk >= p_mj - M
            
            c_disj_1 = solver.Constraint(p_mj[node_j] - BIG_M, infinity, f'seq_{m}_{j}_{k}')
            c_disj_1.SetCoefficient(y[node_k], 1)        # y_mk
            c_disj_1.SetCoefficient(y[node_j], -1)       # - y_mj
            c_disj_1.SetCoefficient(x[x_mjk_key], -BIG_M) # - M * x_mjk
            
            # Constraint 2: If k -> j (x_mkj=1), then y_mj >= y_mk + p_mk
            # Formula: y_mj >= y_mk + p_mk - M * (1 - x_mkj)
            # Rearranged: y_mj - y_mk - M * x_mkj >= p_mk - M
            
            c_disj_2 = solver.Constraint(p_mj[node_k] - BIG_M, infinity, f'seq_{m}_{k}_{j}')
            c_disj_2.SetCoefficient(y[node_j], 1)        # y_mj
            c_disj_2.SetCoefficient(y[node_k], -1)       # - y_mk
            c_disj_2.SetCoefficient(x[x_mkj_key], -BIG_M) # - M * x_mkj

    # 3. Makespan Constraints
    # Cmax >= y_mj + p_mj for all (m,j) in N
    for node in N:
        c_span = solver.Constraint(p_mj[node], infinity, f'span_{node}')
        c_span.SetCoefficient(c_max, 1)
        c_span.SetCoefficient(y[node], -1)

    # --- Objective ---
    objective = solver.Objective()
    objective.SetCoefficient(c_max, 1)
    objective.SetMinimization()

    # ==========================================
    # 4. DYNAMIC HEURISTIC (Transformation Logic)
    # ==========================================
    @lru_cache(maxsize=10000)
    def h_lp_relaxation_standard(state):
        unscheduled = state[unscheduled_var]
        
        # Optimization: Quick exit
        if not unscheduled: return 0.0 

        # Loop through all operations to update bounds
        # Transformation: op_id (DIDP) -> (m, j) (Standard LP)
        for op_id in range(num_ops):
            # 1. Transform op_id to (m, j)
            m = op_machine[op_id]
            j = op_job[op_id]
            node_key = (m, j)
            
            # 2. Update Bounds based on State
            if not state[unscheduled_var].contains(op_id):
                # CASE 1: Operation is FINISHED
                # LB_{mj} = c_(m,j) - p_{mj}
                # Fix variable to historical start time
                actual_completion = state[op_c_vars[op_id]]
                actual_start = actual_completion - p_mj[node_key]
                
                y[node_key].SetBounds(actual_start, actual_start)
            
            else:
                # CASE 2: Operation is UNSCHEDULED
                # LB_{mj} = at_m
                # The start time must be at least the machine's current availability
                machine_ready = state[machine_at_vars[m]]
                y[node_key].SetBounds(machine_ready, infinity)

        # Solve
        status = solver.Solve()
        
        if status == pywraplp.Solver.OPTIMAL:
            return float(objective.Value())
        return 0.0

    return h_lp_relaxation_standard

In [ ]:
def dual_bound_expression_function(didp_bundle):
    """
    Returns a dictionary of heuristic functions (dual bounds) for the JSP model.
    Implements:
    1. Job-based Bound (LB_job)
    2. Machine-based Bound (LB_machine)
    3. One-Machine Preemptive Bound (LB_1mach_preemptive) - Admissible
    4. One-Machine Non-Preemptive Bound (LB_1mach_non_preemptive) - Heuristic
    """
    model, metadata = didp_bundle
    
    # 1. Extract State Variable References
    unscheduled_var = metadata["unscheduled_operations"]
    machine_at_vars = metadata["machine_available_time"]
    op_c_vars = metadata["op_completion_time"]
    
    # 2. Access Static Global Data
    num_jobs = current_number_of_jobs
    num_machines = current_number_of_machines
    op_job_type = current_op_job_type
    ops_on_machine = current_ops_on_machine
    op_processing_time = current_op_processing_time
    
    # Pre-compute Map: Job -> [Operations]
    job_ops_map = [[] for _ in range(num_jobs)]
    for op_id, job_id in enumerate(op_job_type):
        job_ops_map[job_id].append(op_id)

    # PRE-COMPUTATION: Tails (q_jo)
    # Tail = Sum of processing times of all subsequent operations in same job.
    op_tails = [0.0] * len(op_processing_time)
    for j in range(num_jobs):
        ops = job_ops_map[j]
        current_tail = 0.0
        for i in range(len(ops) - 1, -1, -1):
            op = ops[i]
            op_tails[op] = current_tail
            current_tail += op_processing_time[op]
    
    # =========================================================
    # Bound 1: Job-based Bound
    # =========================================================
    @lru_cache(maxsize=10000)
    def h_job_based(state):
        max_job_bound = 0.0
        for j in range(num_jobs):
            c_j = 0.0
            sum_p = 0.0
            for op in job_ops_map[j]:
                if state[unscheduled_var].contains(op):
                    sum_p += op_processing_time[op]
                else:
                    c_op = state[op_c_vars[op]]
                    if c_op > c_j: c_j = c_op
            
            job_bound = c_j + sum_p
            if job_bound > max_job_bound:
                max_job_bound = job_bound
        return float(max_job_bound)

    # =========================================================
    # Bound 2: Machine-based Bound
    # =========================================================
    @lru_cache(maxsize=10000)
    def h_machine_based(state):
        max_machine_bound = 0.0
        for m in range(num_machines):
            at_m = state[machine_at_vars[m]]
            sum_p = 0.0
            for op in ops_on_machine[m]:
                if state[unscheduled_var].contains(op):
                    sum_p += op_processing_time[op]
            
            machine_bound = at_m + sum_p
            if machine_bound > max_machine_bound:
                max_machine_bound = machine_bound
        return float(max_machine_bound)

    # =========================================================
    # Bound 3: One-Machine Preemptive (Baker's / Preemptive Schrage)
    # Considers r_j, p_j, q_j and allows interruption.
    # Always Admissible (Valid Lower Bound).
    # =========================================================
    @lru_cache(maxsize=10000)
    def h_1mach_preemptive(state):
        max_bound = 0.0
        import heapq
        
        # 1. Dynamic Heads
        op_heads = {}
        for j in range(num_jobs):
            current_job_avail = 0.0
            for op in job_ops_map[j]:
                if state[unscheduled_var].contains(op):
                    op_heads[op] = current_job_avail
                    current_job_avail += op_processing_time[op]
                else:
                    current_job_avail = state[op_c_vars[op]]

        # 2. Solve for each machine
        for m in range(num_machines):
            machine_ready = state[machine_at_vars[m]]
            tasks = [] 
            for op in ops_on_machine[m]:
                if state[unscheduled_var].contains(op):
                    r = max(op_heads[op], machine_ready)
                    p = op_processing_time[op]
                    q = op_tails[op]
                    tasks.append([r, p, q])
            
            if not tasks: continue
            
            tasks.sort(key=lambda x: x[0]) # Sort by Release
            
            time_now = 0.0
            ready_queue = [] # Max-heap on Tail q: (-q, task_idx)
            task_idx = 0
            n_tasks = len(tasks)
            current_machine_bound = 0.0
            active_task_idx = -1 
            
            while task_idx < n_tasks or ready_queue or active_task_idx != -1:
                # Jump time if idle
                if active_task_idx == -1 and not ready_queue and task_idx < n_tasks:
                    time_now = max(time_now, tasks[task_idx][0])
                
                # Release tasks
                while task_idx < n_tasks and tasks[task_idx][0] <= time_now:
                    r, p, q = tasks[task_idx]
                    heapq.heappush(ready_queue, (-q, task_idx))
                    task_idx += 1
                
                # Preemption Check
                if active_task_idx != -1 and ready_queue:
                    current_q = tasks[active_task_idx][2]
                    best_waiting_q = -ready_queue[0][0]
                    if best_waiting_q > current_q:
                        # Preempt: put back
                        heapq.heappush(ready_queue, (-current_q, active_task_idx))
                        active_task_idx = -1
                
                # Pick task
                if active_task_idx == -1 and ready_queue:
                    _, idx = heapq.heappop(ready_queue)
                    active_task_idx = idx
                
                # Run
                if active_task_idx != -1:
                    remaining_p = tasks[active_task_idx][1]
                    if task_idx < n_tasks:
                        dt = min(remaining_p, tasks[task_idx][0] - time_now)
                    else:
                        dt = remaining_p
                        
                    time_now += dt
                    tasks[active_task_idx][1] -= dt
                    
                    if tasks[active_task_idx][1] <= 1e-9:
                        finish_q = tasks[active_task_idx][2]
                        current_machine_bound = max(current_machine_bound, time_now + finish_q)
                        active_task_idx = -1
                
            if current_machine_bound > max_bound:
                max_bound = current_machine_bound
                
        return float(max_bound)

    # =========================================================
    # Bound 4: One-Machine Non-Preemptive (Schrage's / Jackson's)
    # Runs tasks to completion. Tighter but potentially Inadmissible.
    # Good feature for EA to learn from.
    # =========================================================
    @lru_cache(maxsize=10000)
    def h_1mach_non_preemptive(state):
        max_bound = 0.0
        import heapq
        
        # 1. Dynamic Heads (Same as above)
        op_heads = {}
        for j in range(num_jobs):
            current_job_avail = 0.0
            for op in job_ops_map[j]:
                if state[unscheduled_var].contains(op):
                    op_heads[op] = current_job_avail
                    current_job_avail += op_processing_time[op]
                else:
                    current_job_avail = state[op_c_vars[op]]

        for m in range(num_machines):
            machine_ready = state[machine_at_vars[m]]
            tasks = []
            for op in ops_on_machine[m]:
                if state[unscheduled_var].contains(op):
                    r = max(op_heads[op], machine_ready)
                    p = op_processing_time[op]
                    q = op_tails[op]
                    tasks.append((r, p, q)) # Tuple is fine here, no mutation needed
            
            if not tasks: continue
            
            tasks.sort(key=lambda x: x[0]) # Sort by Release
            
            time_now = 0.0
            ready_queue = [] # Max-heap on Tail q: (-q, p, r)
            task_idx = 0
            n_tasks = len(tasks)
            current_machine_bound = 0.0
            
            while task_idx < n_tasks or ready_queue:
                # Jump time if idle
                if not ready_queue and task_idx < n_tasks and time_now < tasks[task_idx][0]:
                    time_now = tasks[task_idx][0]
                
                # Release tasks
                while task_idx < n_tasks and tasks[task_idx][0] <= time_now:
                    r, p, q = tasks[task_idx]
                    heapq.heappush(ready_queue, (-q, p, r))
                    task_idx += 1
                
                if ready_queue:
                    # Pick best (Largest Tail)
                    neg_q, p, r = heapq.heappop(ready_queue)
                    q = -neg_q
                    
                    # RUN TO COMPLETION (Non-preemptive)
                    time_now += p
                    
                    current_machine_bound = max(current_machine_bound, time_now + q)
            
            if current_machine_bound > max_bound:
                max_bound = current_machine_bound
                
        return float(max_bound)

    # =========================================================
    # Bound 5: Shifting Bottleneck (Heuristic / ERD Rule)
    # Reference: [cite: 102-127]
    # =========================================================
    @lru_cache(maxsize=10000)
    def h_simplified_shifting_bottleneck(state):
        # 1. Dynamic Heads (implicitly models Source -> Op path)
        op_heads = {}
        for j in range(num_jobs):
            current_job_avail = 0.0
            for op in job_ops_map[j]:
                if state[unscheduled_var].contains(op):
                    op_heads[op] = current_job_avail
                    current_job_avail += op_processing_time[op]
                else:
                    current_job_avail = state[op_c_vars[op]]

        # 2. Calculate Critical Path (CP)
        cp_val = 0.0
        for op in range(len(op_processing_time)):
            if state[unscheduled_var].contains(op):
                # path_len = Head (from S) + Process + Tail (to T)
                path_len = op_heads[op] + op_processing_time[op] + op_tails[op]
                if path_len > cp_val:
                    cp_val = path_len
        
        if cp_val == 0.0: return 0.0

        # 3. Solve Subproblems via ERD Rule
        max_machine_lateness = 0.0

        for m in range(num_machines):
            machine_ready = state[machine_at_vars[m]]
            tasks = [] 
            for op in ops_on_machine[m]:
                if state[unscheduled_var].contains(op):
                    # Step 3: Update Release Date r' = max(r, at_k)
                    r_prime = max(op_heads[op], machine_ready)
                    tasks.append({
                        'r_prime': r_prime,
                        'p': op_processing_time[op],
                        'q': op_tails[op]
                    })
            
            if not tasks: continue

            # Step 3: Sort by ERD
            tasks.sort(key=lambda x: x['r_prime'])

            current_time = machine_ready
            machine_lateness = -float('inf')

            for task in tasks:
                # Earliest start
                start_time = max(current_time, task['r_prime'])
                completion_time = start_time + task['p']
                current_time = completion_time
                
                # Step 3: Lateness L = Completion + Tail - CP
                lateness = completion_time + task['q'] - cp_val
                if lateness > machine_lateness:
                    machine_lateness = lateness

            # Step 4: Max delay
            if machine_lateness > max_machine_lateness:
                max_machine_lateness = machine_lateness

        return float(cp_val + max(0.0, max_machine_lateness))
    
    # =========================================================
    # Bound 6: LP relaxation
    # Reference: [cite: 102-127]
    # =========================================================
    h_lp_relaxation = create_persistent_lp_relaxation_standard_jsp(didp_bundle)
    
    # Return valid registry with all 4 bounds
    return automatic_creation_of_dual_bounds_registry(locals())

In [10]:
dual_bound_functions_registry = dual_bound_expression_function(creation_of_didp_model_function())
display(dual_bound_functions_registry)

{'h_job_based': <function __main__.dual_bound_expression_function.<locals>.h_job_based(state)>,
 'h_machine_based': <function __main__.dual_bound_expression_function.<locals>.h_machine_based(state)>,
 'h_1mach_preemptive': <function __main__.dual_bound_expression_function.<locals>.h_1mach_preemptive(state)>,
 'h_1mach_non_preemptive': <function __main__.dual_bound_expression_function.<locals>.h_1mach_non_preemptive(state)>,
 'h_simplified_shifting_bottleneck': <function __main__.dual_bound_expression_function.<locals>.h_simplified_shifting_bottleneck(state)>,
 'h_lp_relaxation': <function __main__.create_persistent_lp_relaxation_standard_jsp.<locals>.h_lp_relaxation_standard(state)>}

# **Execution**

In [ ]:
# ==========================================
# 1. EVOLUTIONARY ALGORITHM HYPERPARAMETERS
# ==========================================
POPULATION_SIZE = 25        # Size of the population in each generation
GENERATIONS = 10           # Number of generations to run
MUTATION_RATE = 0.2         # Probability of mutating an individual
CROSSOVER_RATE = 0.8        # Probability of performing crossover
ELITISM_RATE = 0.10
# ==========================================
# 2. OPERATOR PARAMETERS
# ==========================================
# Bounds for the coefficients generated for weighted blocks (e.g., 5.5 * h1)
LB_range_of_constant = 0.0  
UB_range_of_constant = 10.0 
# Depth limits for the RPN trees (used in Ramped Half-and-Half generator)
min_chromosome_length = 2               # Minimum depth of the initial trees
max_chromosome_length = 10               # Maximum depth of the initial trees
# Probability of selecting the best individual in the  tournament selection
# Tournament size for parent selection
tournament_size=random.randint(2, 10)
tournament_probability=0.8
# Mutation: Maximum depth allowed for the *newly generated* subtree during mutation
mutation_max_subtree_depth = random.randint(min_chromosome_length, max_chromosome_length)  # Randomly chosen between 1 and 3
# 1-Point Crossover: Probability of using Homology (matching structure) vs Random fallback
homology_1_point_crossover_probability = 0.5
# Subtree Crossover: Probability of swapping a Function (Branch) vs Terminal (Leaf)
subtree_crossover_probability = 0.9
# Uniform Crossover: Probability of swapping genes at a specific index
uniform_crossover_probability = 0.5
# ==========================================
# 4. OTHER PARAMETERS
# ==========================================
# The Ground Truth optimal cost for the specific problem instance
# Used to calculate fitness (deviation from optimal)
OPTIMAL_COST_REFERENCE= 1165
# Time limit (in seconds) for the DIDP solver to run per chromosome evaluation
SOLVER_TIME_LIMIT = 5 #seconds

In [12]:
# B. Configure Params
params = EAHyperparameters(
    # --- 1. Population ---
    population_size=POPULATION_SIZE,          
    generations=GENERATIONS,
    crossover_rate=CROSSOVER_RATE,
    mutation_rate=MUTATION_RATE,
    elitism_rate=ELITISM_RATE,           

    # --- 2. Ranges & Constraints ---
    lb_range_of_constant=LB_range_of_constant,
    ub_range_of_constant=UB_range_of_constant,
    min_chromosome_length=min_chromosome_length,     
    max_chromosome_length=max_chromosome_length,   

    # --- 3. Operator Specifics ---
    tournament_size=tournament_size,                             
    tournament_probability=tournament_probability,                    
    mutation_max_subtree_depth=random.randint(min_chromosome_length, max_chromosome_length),                
    homology_1_point_crossover_probability=homology_1_point_crossover_probability,    
    subtree_crossover_probability=subtree_crossover_probability,             
    uniform_crossover_probability=uniform_crossover_probability,             

    # --- 4. Problem Specific ---
    reference_point=OPTIMAL_COST_REFERENCE,         
    solver_time_limit=SOLVER_TIME_LIMIT,
    
    # Optional: You can override available operations if needed
    available_operations=["ADD", "SUBTRACT", "MAX", "MIN", "MULTIPLY", "PDIV"]
)
best_ind = evolution_algorithm_execution(
    didp_model_registry=creation_of_didp_model_function,
    dual_bound_expression_function=dual_bound_expression_function,
    params=params
)

print(best_ind)

--- Initialization: Generating Population of size 25 - 10 generations ---
Generating Initial Population at time: Tue Dec 16 12:55:35 2025
-> Seeding: 1 Random Terminal + 1 Simple Subtree
Initial Population Generated at time: Tue Dec 16 12:57:45 2025
Initilization time 130.25095796585083
Initial Best Fitness: 0.32532188841201715
Gen 1: Best Fitness = 0.32532188841201715 | Global Best = 0.32532188841201715
Gen 2: Best Fitness = 0.32532188841201715 | Global Best = 0.32532188841201715
Gen 3: Best Fitness = 0.32532188841201715 | Global Best = 0.32532188841201715
Gen 4: Best Fitness = 0.32532188841201715 | Global Best = 0.32532188841201715
Gen 5: Best Fitness = 0.32532188841201715 | Global Best = 0.32532188841201715
Gen 6: Best Fitness = 0.32532188841201715 | Global Best = 0.32532188841201715
Gen 7: Best Fitness = 0.32532188841201715 | Global Best = 0.32532188841201715
Gen 8: Best Fitness = 0.32532188841201715 | Global Best = 0.32532188841201715
Gen 9: Best Fitness = 0.32532188841201715 | Gl